In [31]:
import pandas as pd
import mysql.connector


df = pd.read_csv(r'C:\Users\RICHA\Desktop\python\data\projects\fraud-detection-analysis\data\creditcard.csv')
print("CSV loaded:", df.shape)


conn = mysql.connector.connect(
    host     = '127.0.0.1',
    port     = 3306,
    user     = 'root',
    password = 'richasql@2006',
    database = 'fraud_db'
)
cursor = conn.cursor()

CSV loaded: (284807, 31)


In [25]:

#A cursor is like a pen that writes SQL commands to the database



cursor.execute("DROP TABLE IF EXISTS transactions")
cursor.execute("""
    CREATE TABLE transactions (
        Time FLOAT, V1 FLOAT, V2 FLOAT, V3 FLOAT, V4 FLOAT,
        V5 FLOAT, V6 FLOAT, V7 FLOAT, V8 FLOAT, V9 FLOAT,
        V10 FLOAT, V11 FLOAT, V12 FLOAT, V13 FLOAT, V14 FLOAT,
        V15 FLOAT, V16 FLOAT, V17 FLOAT, V18 FLOAT, V19 FLOAT,
        V20 FLOAT, V21 FLOAT, V22 FLOAT, V23 FLOAT, V24 FLOAT,
        V25 FLOAT, V26 FLOAT, V27 FLOAT, V28 FLOAT,
        Amount FLOAT, Class INT
    )
""")


print("Loading rows... please wait")
batch_size = 1000
data = df.values.tolist()
# pandas dataframe to plain python lists
placeholders = ','.join(['%s'] * 31)

for i in range(0, len(data), batch_size):
    batch = data[i:i+batch_size]
    cursor.executemany(f"INSERT INTO transactions VALUES ({placeholders})", batch)
    conn.commit()
    if i % 50000 == 0:
        print(f"  Loaded {i:,} rows...")

print("DONE! All", len(df), "rows loaded.")
cursor.close()
conn.close()

CSV loaded: (284807, 31)
Loading rows... please wait
  Loaded 0 rows...
  Loaded 50,000 rows...
  Loaded 100,000 rows...
  Loaded 150,000 rows...
  Loaded 200,000 rows...
  Loaded 250,000 rows...
DONE! All 284807 rows loaded.


In [32]:
def run_query(query):
    """Run any SQL query and return result as a pandas dataframe"""
    cursor.execute(query)
    rows    = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]
    return pd.DataFrame(rows, columns=columns)
print("Ready! Use run_query('your SQL here') for all queries")

Ready! Use run_query('your SQL here') for all queries


In [33]:
result=run_query('SELECT COUNT(*) AS total_rows FROM transactions')
print(result)


split=run_query(
    'SELECT Class, COUNT(*) AS count FROM transactions GROUP BY Class')
print(split)

   total_rows
0      284807
   Class   count
0      0  284315
1      1     492
